# 07 — Structural Breaks & Entropy

**AFML Chapters 14 & 15** — López de Prado (2018)

Financial time series are non-stationary.  Regime changes, bubbles,
and structural breaks invalidate models trained on stale regimes.

This notebook demonstrates:
1. **CUSUM filter** — event-driven sampling that detects mean-shifts
2. **SADF (Supremum ADF)** — tests for explosive / bubble behaviour
3. **Entropy measures** — Shannon, plug-in, and Lempel-Ziv complexity

---

In [ ]:
from _setup import *
from afml.labeling import daily_volatility
from afml.features import (
    cusum_filter,
    sadf,
    shannon_entropy,
    lempel_ziv_complexity,
    encode_returns_binary,
)

## 1. CUSUM filter

Instead of sampling at fixed intervals (time bars), the CUSUM filter
fires events only when the price deviates enough from its running mean.
This produces adaptive sampling: more events during volatile periods,
fewer during calm periods.

In [ ]:
panel = load_daily_bars()
btc = panel[panel["symbol"] == "BTC-USD"].sort_values("ts").drop_duplicates("ts", keep="last")
close = btc.set_index("ts")["close"]

vol = daily_volatility(close, span=20)
threshold = vol.mean() * 2  # 2× average daily vol

events = cusum_filter(close, threshold=threshold * close.mean())
print(f"CUSUM threshold: {threshold:.4f} (2× avg daily vol)")
print(f"Events: {len(events)} out of {len(close)} days ({len(events)/len(close):.1%} sampled)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(close, color=NAVY, lw=0.8, label="BTC close")
ax.scatter(events, close.reindex(events), color=RED, s=15, zorder=5, label=f"CUSUM events ({len(events)})")
ax.set_ylabel("Price (USD)")
ax.set_title("CUSUM filter: event-driven sampling on BTC", fontweight="bold")
ax.legend(fontsize=9)

# Event density over time
ax = axes[1]
event_density = pd.Series(1, index=events).resample("Q").count()
ax.bar(event_density.index, event_density.values, width=80, color=TEAL, alpha=0.7)
ax.set_ylabel("Events per quarter")
ax.set_title("Event density: more events during volatile regimes", fontweight="bold")

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "07_cusum_filter.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2. Threshold sensitivity

In [ ]:
multipliers = [0.5, 1, 2, 3, 5, 10]
base = vol.mean() * close.mean()

results = []
for m in multipliers:
    ev = cusum_filter(close, threshold=base * m)
    results.append({"multiplier": m, "n_events": len(ev), "pct_sampled": len(ev)/len(close)*100})

df_sens = pd.DataFrame(results)
print(df_sens.to_string(index=False, float_format="%.1f"))

## 3. SADF — detecting explosive behaviour

The Supremum ADF test runs expanding-window ADF tests and takes the
maximum.  SADF > critical value (≈ 0 for large samples) signals
explosive / bubble-like dynamics.

In [ ]:
# Use a subset for speed (SADF is O(n²))
close_recent = close.loc["2020":]
log_p = np.log(close_recent)

print(f"Running SADF on {len(log_p)} observations (2020+)...")
sadf_result = sadf(log_p, min_window=60, max_window=252, lags=1)
print(f"SADF computed: {len(sadf_result)} dates")
print(f"Max SADF stat: {sadf_result['sadf_stat'].max():.2f}")
print(f"Min SADF stat: {sadf_result['sadf_stat'].min():.2f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(close_recent, color=NAVY, lw=0.8)
ax.set_ylabel("Price (USD)")
ax.set_title("BTC price (2020+)", fontweight="bold")

ax = axes[1]
ax.plot(sadf_result.index, sadf_result["sadf_stat"], color=TEAL, lw=0.8)
ax.axhline(0, color=RED, ls="--", lw=1, label="Critical value (≈0)")
ax.fill_between(sadf_result.index, 0, sadf_result["sadf_stat"],
                where=sadf_result["sadf_stat"] > 0, color=RED, alpha=0.2,
                label="Explosive regime")
ax.set_ylabel("SADF statistic")
ax.set_title("Supremum ADF: detecting explosive behaviour", fontweight="bold")
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "07_sadf.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Entropy measures

Entropy quantifies the information content (randomness) of a series.
Low entropy → predictable, high entropy → random.  Useful as a feature
to detect regime changes.

In [ ]:
symbols = ["BTC-USD", "ETH-USD", "SOL-USD", "DOGE-USD", "XRP-USD"]
entropy_results = []

for sym in symbols:
    asset = panel[panel["symbol"] == sym].sort_values("ts").drop_duplicates("ts", keep="last")
    c = asset.set_index("ts")["close"]
    rets = np.log(c / c.shift(1)).dropna()

    binary = encode_returns_binary(rets)

    entropy_results.append({
        "symbol": sym,
        "shannon": shannon_entropy(rets, n_bins=30),
        "lz_complexity": lempel_ziv_complexity(binary),
        "n_obs": len(rets),
    })

df_ent = pd.DataFrame(entropy_results).set_index("symbol")
display(df_ent.style.format({"shannon": "{:.3f}", "lz_complexity": "{:.3f}", "n_obs": "{:,}"}))

In [ ]:
# Rolling entropy for BTC
rets_btc = np.log(close / close.shift(1)).dropna()
window = 60

rolling_shannon = rets_btc.rolling(window).apply(
    lambda x: shannon_entropy(x, n_bins=15), raw=False
)
rolling_lz = rets_btc.rolling(window).apply(
    lambda x: lempel_ziv_complexity(encode_returns_binary(x)), raw=False
)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

ax = axes[0]
ax.plot(close, color=NAVY, lw=0.8)
ax.set_ylabel("Price")
ax.set_title("BTC price", fontweight="bold")

ax = axes[1]
ax.plot(rolling_shannon.index, rolling_shannon, color=TEAL, lw=0.8)
ax.set_ylabel("Shannon entropy")
ax.set_title(f"Rolling Shannon entropy ({window}d)", fontweight="bold")

ax = axes[2]
ax.plot(rolling_lz.index, rolling_lz, color=GOLD, lw=0.8)
ax.set_ylabel("LZ complexity")
ax.set_title(f"Rolling Lempel-Ziv complexity ({window}d)", fontweight="bold")

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "07_entropy.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Summary

**Structural Breaks (Ch 14):**
- CUSUM filter adaptively samples events — more during volatile regimes, fewer during calm
- SADF detects explosive (bubble-like) dynamics by finding the supremum of expanding-window ADF tests
- Both can serve as features or event filters for ML models

**Entropy (Ch 15):**
- Shannon entropy and Lempel-Ziv complexity measure how "random" a return series is
- Low entropy periods may be more predictable — useful as a feature or filter
- Rolling entropy reveals regime changes in information content

**For production use:**
- Use CUSUM events as the trigger for triple-barrier labeling (instead of every bar)
- Include SADF and rolling entropy as features in ML models
- Monitor SADF for bubble detection / risk management

---

*Next: [08_microstructure.ipynb](08_microstructure.ipynb) — Market Microstructure (AFML Ch 16)*